# 07 · Inférence sur de grandes images avec SAHI

**Objectif.** Appliquer `best.pt` à des images entières (plusieurs milliers de pixels de côté). SAHI découpe l'image en tuiles de 640 × 640
avec 20 % de recouvrement, prédit sur chaque tuile, puis fusionne les détections. Une figure est produite par classe détectée.

**Entrées.** `best.pt` ; images xView 122 (`122.tif`, `122_2.tif`), issues du lot `val_images` de xView : absentes de l'entraînement, sans annotation publique.

**Sorties.** Comptage des détections par classe (sorties texte conservées) et une figure par classe.
Exemples exportés : `results/figures/predictions_xview122_*.jpg`.

*Exécuté sur Kaggle (GPU) ; les 14 figures ont été retirées des sorties (22 Mo).*

In [ ]:
# --- Configuration des chemins ---
# Exécution d'origine sur Kaggle : KAGGLE_INPUT = /kaggle/input/datasets, WORK_DIR = /kaggle/working.
# En local, définir les variables d'environnement GOD_EYE_INPUT et GOD_EYE_WORK (ou modifier les valeurs ci-dessous).
import os
KAGGLE_INPUT = os.environ.get("GOD_EYE_INPUT", "/kaggle/input/datasets")
WORK_DIR = os.environ.get("GOD_EYE_WORK", "/kaggle/working")

# Poids du modèle final (best.pt). Sur Kaggle : sauvegarde de l'entraînement publiée comme dataset.
# En local : télécharger best.pt depuis la Release GitHub et définir GOD_EYE_WEIGHTS=chemin/vers/best.pt
MODEL_PATH = os.environ.get("GOD_EYE_WEIGHTS", f"{KAGGLE_INPUT}/ayiamavigan/xview-dota-ckpt-backup/run_state/weights/best.pt")

import torch
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

In [ ]:
%pip install -q ultralytics sahi

In [11]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import visualize_object_predictions


# -------------------------------------------------------------------
# Chemins
# -------------------------------------------------------------------

# MODEL_PATH et DEVICE : définis dans la cellule de configuration

# Images xView 122 (lot val_images de xView : jamais vues à l'entraînement)
IMAGE_PATHS = [
    f"{KAGGLE_INPUT}/ayiamavigan/test-comp-model/122.tif",
    f"{KAGGLE_INPUT}/ayiamavigan/test-comp-model/122_2.tif",
]
# Autre test réalisé : dataset ayiamavigan/2-pic-btr-test (brt0.png, brt1.png)

# 0.0 affiche toutes les prédictions, y compris le bruit.
# Pour un résultat plus propre, essaie ensuite 0.25.
CONFIDENCE_THRESHOLD = 0.25


# -------------------------------------------------------------------
# Chargement du modèle
# -------------------------------------------------------------------

detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=MODEL_PATH,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    device=DEVICE,
)


# -------------------------------------------------------------------
# Inférence et affichage par classe
# -------------------------------------------------------------------

for image_path in IMAGE_PATHS:
    image_name = Path(image_path).name

    print("\n" + "=" * 90)
    print(f"Image : {image_name}")

    result = get_sliced_prediction(
        image_path,
        detection_model,
        slice_height=640,
        slice_width=640,
        overlap_height_ratio=0.20,
        overlap_width_ratio=0.20,
    )

    predictions = result.object_prediction_list

    # Comptage global
    labels = [pred.category.name for pred in predictions]
    counts = Counter(labels)

    print(f"Nombre total de détections : {len(predictions)}")

    if not counts:
        print("Aucune détection.")
        continue

    print("Classes détectées :")
    for label, count in counts.most_common():
        print(f"  {label:25s} : {count}")

    image_np = np.array(Image.open(image_path).convert("RGB"))

    # Une sortie séparée pour chaque classe
    for label, count in counts.most_common():

        class_predictions = [
            pred
            for pred in predictions
            if pred.category.name == label
        ]

        scores = [
            pred.score.value
            for pred in class_predictions
        ]

        # Visualisation avec seulement cette classe
        vis = visualize_object_predictions(
                image=image_np,
                object_prediction_list=class_predictions,
                hide_labels=True,
                hide_conf=True,
                output_dir=None,
                file_name=None,
            )

        score_max = max(scores) if scores else 0.0
        score_moyen = sum(scores) / len(scores) if scores else 0.0

        plt.figure(figsize=(16, 10))
        plt.imshow(vis["image"])
        plt.title(
            f"{image_name} — classe : {label} — "
            f"{count} détection(s) — "
            f"score max : {score_max:.3f} — "
            f"score moyen : {score_moyen:.3f}"
        )
        plt.axis("off")
        plt.show()


Image : 122.tif
Performing prediction on 30 slices.


[ WARN:0@1862.853] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 33550 (0x830e) encountered
[ WARN:0@1862.853] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 33922 (0x8482) encountered
[ WARN:0@1862.853] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 34735 (0x87af) encountered
[ WARN:0@1862.853] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 34736 (0x87b0) encountered
[ WARN:0@1862.853] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 34737 (0x87b1) encountered
[ WARN:0@1862.853] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 42112 (0xa480) encountered
[ WARN:0@1862.853] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 42113 (0xa481) encountered


Nombre total de détections : 1556
Classes détectées :
  small-vehicle             : 812
  building                  : 561
  large-vehicle             : 86
  ship                      : 61
  shipping-container        : 27
  engineering-vehicle       : 5
  vehicle-lot               : 4



Image : 122_2.tif
Performing prediction on 30 slices.
Nombre total de détections : 1452
Classes détectées :
  small-vehicle             : 757
  building                  : 546
  large-vehicle             : 72
  ship                      : 60
  shipping-container        : 11
  vehicle-lot               : 4
  engineering-vehicle       : 2
